In [8]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("acmecorp-employee-handbook.pdf")
data = loader.load()
print(data)

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment free from

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)
print(len(all_splits))

3


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 10594.53it/s]


In [12]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

In [13]:
ids = vector_store.add_documents(documents=all_splits)

In [14]:
results = vector_store.similarity_search("How many days of vacation does an employee get in their first year?")
print(results[0])

page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

In [19]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for company policies, vacation days, and other employee information."""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [20]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama
# Initialiser le modèle Ollama
model = ChatOllama(
    model="llama3.2:3b", # ou mistral, gemma, etc.
    temperature=0
)
agent = create_agent(
    model=model,
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee hand-book for information."
)

response = agent.invoke({"messages": [HumanMessage(content="How many days of vacation does an em-ployee get in their first year?")]})
print(response['messages'][-1].content)

{"name": "Search the employee handbook", "parameters": {"query": {"type": "string"}}}


In [23]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [24]:
from langchain.tools import tool
@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        print(f"Executing SQL query: {query}")
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [25]:
sql_query.invoke("SELECT * FROM Artist LIMIT 10")

Executing SQL query: SELECT * FROM Artist LIMIT 10


"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [27]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama
# Initialiser le modèle Ollama
model = ChatOllama(
    model="llama3.2:3b", # ou mistral, gemma, etc.
)
system_prompt = """You are a SQL expert.
    Rules:
    - Only use sql_query tool
    - The sql_query tool takes a SQL query as input and returns the result of the
    query.
    - Only use available columns
    - If information does not exist, say so
    - Do not guess
    - you have to return the results in a human readable format, do not return raw
    SQL results or a sql query.
    Database schema:
    Table Artist:
    - ArtistId
    - Name
"""
agent = create_agent(
    model=model,
    tools=[sql_query],
    system_prompt=system_prompt
)

In [28]:
question = HumanMessage(content="Give me the first 5 artists in the database")
response = agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

{"name": "Obtain information from the database using SQL queries", "parameters": {"query": {"type": "string", "query": "SELECT ArtistId, Name FROM Artist LIMIT 5"}}}
